<a href="https://colab.research.google.com/github/kritika-pandey/JSONObjectStore/blob/main/JSON_Object_Store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Objective: Build a small, homemade database library (jdb) that saves Python objects to a JSON file on disk and loads them back as working objects, then prove it works using vehicle objects

In [5]:
# ============================================================
# CELL 1: The "jdb" mini database library
# Purpose: Define everything needed to save Python objects to a
# JSON file and load them back. Run this cell FIRST.
# ============================================================

import os      # used to check whether the database file already exists
import json    # used to convert dictionaries to/from JSON text


# ---------- Validation helpers (from utils.py) ----------
# Small functions that check inputs, so mistakes are caught early
# with a clear error message.

def _valid_name_(name):
    # Table names must be strings, otherwise stop with an error
    if not isinstance(name, str):
        raise TypeError("Table name must be a string")


def _valid_key_(key):
    # Keys must be strings (JSON file keys can only be strings)
    if not isinstance(key, str):
        raise TypeError("Key name must be a string")


# ---------- Base class (from db.py) ----------
# Handles loading and saving the file. KVStore and DBTable both
# inherit from this, so the file logic is written only once.

class Database(object):
    def __init__(self, location=None):
        if location:
            # A file path was given: remember it and load existing data
            self.db_location = location
            self._load_()
        else:
            # No path given: start with an empty in-memory dictionary
            self.db = {}

    def _load_(self):
        # If the file exists from an earlier run, read it into self.db.
        # If not (first run), start with an empty dictionary.
        if os.path.exists(self.db_location):
            db_file = open(self.db_location, "r")
            self.db = json.load(db_file)    # JSON text -> Python dictionary
            db_file.close()
        else:
            self.db = {}

    def save(self, location=None):
        # Choose where to write: a path passed to save() wins,
        # otherwise use the path given when the object was created
        if location:
            db_file = open(location, "w")
        elif self.db_location:
            db_file = open(self.db_location, "w")
        else:
            raise ValueError("DB location is null")

        json.dump(self.db, db_file)         # Python dictionary -> JSON text in file
        db_file.close()                     # close so data is fully written to disk


# ---------- Key-value storage (from kv.py) ----------
# A flat store: each key maps to one saved object.

class KVStore(Database):
    def set(self, key, value):
        _valid_key_(key)
        # JSON can't store a custom object, so save its attributes
        # (value.__dict__) as a JSON string instead
        self.db[key] = json.dumps(value.__dict__)

    def get(self, key):
        _valid_key_(key)
        # Turn the saved JSON string back into a dictionary
        return json.loads(self.db[key])

    def delete(self, key):
        _valid_key_(key)
        if key in self.db:      # deleting a missing key does nothing
            del self.db[key]

    def exists(self, key):
        _valid_key_(key)
        return key in self.db   # True or False

    def keys(self):
        return self.db.keys()   # all stored keys, for looping

    def count(self):
        return len(self.db)     # number of stored entries

    def key_count(self):
        # Added because KVApp calls key_count(); the original only had count()
        return self.count()


# ---------- Table storage (from table.py) ----------
# Same idea as KVStore, but entries live inside a named table:
# self.db["CARS"]["h1"] instead of self.db["h1"]

class DBTable(Database):
    def __init__(self, table_name, location=None):
        super().__init__(location)   # run Database.__init__ (load the file)
        self._create_(table_name)    # then set up the table

    def _create_(self, name):
        _valid_name_(name)
        self.db[name] = {}           # empty dictionary to hold this table's rows
        self.table_name = name       # remember which table we are working on

    def insert(self, key, value):
        _valid_key_(key)
        # Same trick as KVStore.set, stored one level deeper
        self.db[self.table_name][key] = json.dumps(value.__dict__)

    def select(self, key):
        _valid_key_(key)
        return self.db[self.table_name][key]     # one row by key

    def delete(self, key):
        _valid_key_(key)
        del self.db[self.table_name][key]

    def exists(self, key):
        _valid_key_(key)
        return key in self.db[self.table_name]

    def count(self):
        return len(self.db[self.table_name])     # number of rows in the table

    def select_all(self):
        return self.db[self.table_name]          # every row in the table

In [2]:
# ============================================================
# CELL 2: The "vehicles" classes
# Purpose: Create objects with real, changing data (state and
# speed) to use as test data for the database. Run after Cell 1.
# ============================================================

class Car:
    brand = "MG Hector"            # class attribute: shared by all Car objects

    @classmethod
    def launch_year(cls):
        return 2020                # class method: works without creating an object

    def __init__(self, current_state="Standing", current_speed=0):
        # Every new car starts standing still unless told otherwise.
        # The defaults also let us rebuild a car from saved data:
        # Car(**saved_dictionary)
        self.current_state = current_state
        self.current_speed = current_speed

    def start(self):
        self.current_state = "Running"
        self.current_speed = 10    # a car that starts moves at speed 10

    def stop(self):
        self.current_state = "Standing"
        self.current_speed = 0

    def speed_up(self, speed):
        if self.current_state == "Standing":
            self.start()           # a standing car must start before speeding up
        self.current_speed += speed

    def speed_down(self, speed):
        if self.current_state == "Running":
            self.current_speed -= speed   # only a running car can slow down

    def show_state(self):
        print(f"{self.current_state} and current speed is {self.current_speed}")


class Hector(Car):
    # Inherits everything from Car (start, stop, speed_up, ...).
    # Only overrides show_state to print a Hector-specific message.
    def show_state(self):
        print(f"Hector is {self.current_state} and current speed is {self.current_speed}")

In [6]:
# ============================================================
# CELL 3: Key-value store demo (KVApp)
# Purpose: Prove that KVStore can save objects, reload them as
# working objects, list them and delete them.
# Needs Cells 1 and 2 to have been run.
# ============================================================

def get_hector(db):
    # db.get("Hector") returns a dictionary like
    #   {"current_state": "Running", "current_speed": 50}
    # The ** unpacks it into Hector(current_state=..., current_speed=...),
    # which rebuilds a real, working Hector object.
    return Hector(**db.get("Hector"))


# Step 1: create two vehicles and change their state, so there is data to save
h1 = Hector()
h1.speed_up(40)       # starts (speed 10), then +40 -> speed 50

c1 = Car()
c1.speed_up(20)       # starts (speed 10), then +20 -> speed 30

# Step 2: create the store; loads "my-kv_store" if it exists, else starts empty
store = KVStore("my-kv_store")

# Step 3: put both objects in the store (in memory only, so far)
store.set("Hector", h1)
store.set("Car", c1)

# Step 4: write the store to disk
store.save()

# Step 5: read "Hector" back and rebuild a real object to prove it works
h2 = get_hector(store)
h2.show_state()       # Hector is Running and current speed is 50

# Step 6: how many entries are stored? (2)
print(str(store.key_count()))

# Step 7: print every key and its saved value
for key in store.keys():
    print("Key: {}, Value: {}".format(key, store.get(key)))

# Step 8: delete one entry (in memory only; save() was not called again)
store.delete("Hector")

# Step 9: print again to prove "Hector" is gone and only "Car" remains
for key in store.keys():
    print("Key: {}, Value: {}".format(key, store.get(key)))

Hector is Running and current speed is 50
2
Key: Hector, Value: {'current_state': 'Running', 'current_speed': 50}
Key: Car, Value: {'current_state': 'Running', 'current_speed': 30}
Key: Car, Value: {'current_state': 'Running', 'current_speed': 30}


In [7]:
# ============================================================
# CELL 4: Table store demo (TableApp)
# Purpose: Prove that DBTable can store several objects under
# one named table and save them to disk.
# Needs Cells 1 and 2 to have been run.
# ============================================================

# Step 1: create three Hectors with different speeds.
# Named t1, t2, t3 so they don't overwrite h1 and h2 from Cell 3.
t1 = Hector(); t1.speed_up(40)    # speed 50
t2 = Hector(); t2.speed_up(20)    # speed 30
t3 = Hector(); t3.speed_up(60)    # speed 70

# Step 2: create a table named "CARS", saved to the file "my-cars"
cars_table = DBTable("CARS", "my-cars")

# Step 3: insert each object under its own key inside the table
cars_table.insert("h1", t1)
cars_table.insert("h2", t2)
cars_table.insert("h3", t3)

# Step 4: print the whole dictionary to see the nested structure:
# {'CARS': {'h1': ..., 'h2': ..., 'h3': ...}}
print(cars_table.db)

# Step 5: write everything to disk
cars_table.save()

{'CARS': {'h1': '{"current_state": "Running", "current_speed": 50}', 'h2': '{"current_state": "Running", "current_speed": 30}', 'h3': '{"current_state": "Running", "current_speed": 70}'}}
